In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/AIKONIC-AI/PHASE_02

import sys, os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score

if os.getcwd() not in sys.path: sys.path.insert(0, os.getcwd())
import config
from data_loader import DataLoader

loader = DataLoader()

# Load Original Unaugmented Images ONLY
def _load_originals(loader_instance, split_name):
    df = loader_instance._train_df if split_name == "train" else loader_instance._val_df
    if "augmented" in df.columns:
        df = df[df["augmented"] == False].reset_index(drop=True)
    from data_loader import _load_image
    paths = df["image_path"].astype(str).tolist()
    y = df["label"].astype(int).to_numpy()
    xs = [_load_image(tf.constant(p)).numpy() for p in paths]
    return np.stack(xs, axis=0).astype("float32"), y

X_train_orig, y_train_orig = _load_originals(loader, "train")
X_val_orig, y_val_orig = _load_originals(loader, "val")
X_cv = np.concatenate([X_train_orig, X_val_orig], axis=0)
y_cv = np.concatenate([y_train_orig, y_val_orig], axis=0)

# CV Model Builder
def build_cv_model():
    inputs = tf.keras.Input(shape=(224, 224, 1))
    x = tf.keras.layers.Concatenate(axis=-1)([inputs, inputs, inputs])

    # 1. Start with a FROZEN base so it doesn't destroy ImageNet weights
    base = tf.keras.applications.MobileNetV3Small(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
    base.trainable = False
    x = base(x, training=False)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dense(config.DENSE_UNITS, activation=config.ACTIVATION)(x)
    x = tf.keras.layers.Dropout(config.DROPOUT_RATE)(x)
    outputs = tf.keras.layers.Dense(config.NUM_CLASSES, activation="softmax")(x)

    m = tf.keras.Model(inputs, outputs)

    # 2. Use PHASE_A_LR (1e-4) instead of PHASE_B_LR (1e-6) so it actually learns
    m.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=config.PHASE_A_LR),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return m

# Run 5-Fold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

skf = StratifiedKFold(n_splits=config.CV_FOLDS, shuffle=True, random_state=config.RANDOM_SEED)
results = []

for fold, (tr_idx, vl_idx) in enumerate(skf.split(X_cv, y_cv), 1):
    print(f"\nRunning Fold {fold}/{config.CV_FOLDS}..." + "─"*30)

    # Build a fresh brain
    model = build_cv_model()
    y_tr_cat = tf.keras.utils.to_categorical(y_cv[tr_idx], 2)
    y_vl_cat = tf.keras.utils.to_categorical(y_cv[vl_idx], 2)

    # Train
    model.fit(X_cv[tr_idx], y_tr_cat, validation_data=(X_cv[vl_idx], y_vl_cat),
              epochs=config.CV_EPOCHS, batch_size=config.BATCH_SIZE, verbose=0)

    # Predict on the unseen Validation chunk
    probs = model.predict(X_cv[vl_idx], verbose=0)
    preds = np.argmax(probs, axis=1)

    # Calculate ALL 5 Metrics
    acc = accuracy_score(y_cv[vl_idx], preds)
    prec = precision_score(y_cv[vl_idx], preds, zero_division=0)
    sens = recall_score(y_cv[vl_idx], preds, zero_division=0)
    f1 = f1_score(y_cv[vl_idx], preds, zero_division=0)
    auroc = roc_auc_score(y_cv[vl_idx], probs[:, 1]) # Looks at the specific PD probability

    # Calculate Confusion Matrix
    cm = confusion_matrix(y_cv[vl_idx], preds)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)

    # Save the data
    results.append({
        "fold": fold,
        "accuracy": acc,
        "precision": prec,
        "sensitivity": sens,
        "f1_score": f1,
        "auroc": auroc,
        "TP": tp, "TN": tn, "FP": fp, "FN": fn
    })

    # Print the detailed breakdown for this fold
    print(f"  Acc: {acc:.4f} | Sens: {sens:.4f} | Prec: {prec:.4f} | F1: {f1:.4f} | AUC: {auroc:.4f}")
    print(f"  CM : TP={tp} TN={tn} FP={fp} FN={fn}")

# Save Results & Calculate Averages
cv_df = pd.DataFrame(results)

# Calculate the Mean and Standard Deviation for all metric columns
mean_row = {k: (cv_df[k].mean() if k != 'fold' else 'mean') for k in cv_df.columns}
std_row = {k: (cv_df[k].std() if k != 'fold' else 'std') for k in cv_df.columns}

# Append the summary rows to the bottom of the dataframe
cv_df = pd.concat([cv_df, pd.DataFrame([mean_row, std_row])], ignore_index=True)

# Save to your evaluation folder
cv_path = os.path.join(config.EVAL_DIR, "cv_results_detailed.csv")
cv_df.to_csv(cv_path, index=False)

print("\n" + "="*55)
print(" FINAL 5-FOLD CROSS-VALIDATION SUMMARY")
print("="*55)
print(f" Mean Accuracy   : {mean_row['accuracy']:.4f} ± {std_row['accuracy']:.4f}")
print(f" Mean Precision  : {mean_row['precision']:.4f} ± {std_row['precision']:.4f}")
print(f" Mean Sensitivity: {mean_row['sensitivity']:.4f} ± {std_row['sensitivity']:.4f}")
print(f" Mean F1-Score   : {mean_row['f1_score']:.4f} ± {std_row['f1_score']:.4f}")
print(f" Mean AUROC      : {mean_row['auroc']:.4f} ± {std_row['auroc']:.4f}")
print(f"✓ Detailed results saved to {cv_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1TTDXrdoUkLETneSUJKezS9PEaKVKjYHh/AIKONIC-AI/PHASE_02

Running Fold 1/5...──────────────────────────────
4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
  Acc: 0.8652 | Sens: 0.8000 | Prec: 0.8889 | F1: 0.8421 | AUC: 0.9101
  CM : TP=64 TN=90 FP=8 FN=16

Running Fold 2/5...──────────────────────────────
  Acc: 0.8202 | Sens: 0.6875 | Prec: 0.8871 | F1: 0.7746 | AUC: 0.8832
  CM : TP=55 TN=91 FP=7 FN=25

Running Fold 3/5...──────────────────────────────


  Acc: 0.8418 | Sens: 0.7595 | Prec: 0.8696 | F1: 0.8108 | AUC: 0.9000
  CM : TP=60 TN=89 FP=9 FN=19

Running Fold 4/5...──────────────────────────────


  Acc: 0.8192 | Sens: 0.7468 | Prec: 0.8310 | F1: 0.7867 | AUC: 0.8613
  CM : TP=59 TN=86 FP=12 FN=20

Running Fold 5/5...──────────────────────────────
  Acc: 0.8475 | Sens: 0.7595 | Prec: 0.8824 | F1: 0.8163 | AUC: 0.8897
  CM : TP=60 TN=90 FP=8 FN=19

 📊 FINAL 5-FOLD CROSS-VALIDATION SUMMARY
 Mean Accuracy   : 0.8388 ± 0.0194
 Mean Sensitivity: 0.7507 ± 0.0406
 Mean Precision  : 0.8718 ± 0.0240
 Mean F1-Score   : 0.8061 ± 0.0264
 Mean AUROC      : 0.8888 ± 0.0185
✓ Detailed results saved to /content/drive/.shortcut-targets-by-id/1TTDXrdoUkLETneSUJKezS9PEaKVKjYHh/AIKONIC-AI/evaluation/cv_results_detailed.csv
